In [1]:
# Course notation: find Latex_macros.ipynb in this folder or in any parent folder, so the cell works wherever the notebook sits
import pathlib
_here = pathlib.Path.cwd().resolve()
_macros = next((d / "Latex_macros.ipynb" for d in [_here, *_here.parents] if (d / "Latex_macros.ipynb").exists()), None)
if _macros is not None:
    get_ipython().run_line_magic("run", f'"{_macros}"')
else:
    print("Latex_macros.ipynb not found: math may show without the course notation")

<IPython.core.display.Latex object>

# Putting it together: the landscape of reinforcement learning methods

## Where this session fits

> **The question we start from: how do all these methods fit together, and how do we choose among them for a new problem?**

- **Position: the closing session, no new method**
    - Part A (A1 to A6) built the value-based path (Bellman and dynamic programming, sampling, control, deep Q-learning)
    - Part B (B1 to B4) built the policy-gradient methods
        - B1 the theorem, B2 the weight and the baseline, B3 the critic, B4 how far one update goes
    - Part C (C1, C2) applied them to language models: PPO on a language model, and GRPO
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. place any method of the series by three questions: is a model needed, what does it learn, which policy generates its data
2. read the threads of the series across all three parts
    - bias and variance, backup width and depth, on- and off-policy
    - evaluate versus improve, exploration
    - in Part C, the proxy problem
3. say which equation carries each idea of the series
4. choose a method family for a new problem, and say what the series left out


# Three questions place every method

**Why this section?** There are many methods. Three questions give each one an address.

- **Is a model needed?**
    - *model-based:* a model of the transitions and rewards is used for planning (known here: value and policy iteration)
    - *model-free:* only sampled experience (everything from Monte Carlo on)
- **What does it learn?**
    - *values:* $\statevalfun$ or $\actvalfun$; the policy is read off (the greedy action)
    - *a policy:* $\pi_\theta$ directly, by gradient ascent (Part B)
    - *both:* actor-critic and PPO learn a policy and a value (the critic is the learned value)
- **Which policy generates the data?**
    - *on-policy:* the policy being learned (SARSA, REINFORCE)
    - PPO and GRPO are *near* on-policy (a slightly older copy, corrected by the ratio)
    - *off-policy:* another policy, or fixed data (Q-learning, DQN)


**The grid**

<img src="images/methods_grid.png" width=80%>

- the rows combine two questions: with a known model there is no sampled data, so the model-based row has no on-/off-policy split
- the value-based path (green and blue) sits in the left column
- Part B (orange) in the middle and right
- Part C's PPO and GRPO are in the same cells as their Part B relatives
- two kinds of empty cell
    - *none here*: no model-based policy method in this series
    - *not covered*: off-policy actor-critic methods exist, but the series skips them
- the three questions are listed in the order values, both, policy in the columns


## Table 1, extended: all the methods

**Why this section?** Table 1 (the value-based methods, end of Part A) condensed, then the Part B and Part C rows.

> **The depth and bias columns carry over from Part A to the weight of the policy gradient: the same dial.**

**Terms from earlier sessions**

- *p*-step TD: look $p$ steps ahead
- GAE (generalized advantage estimation): a blend of depths
- critic: the learned value used by an actor
- ratio: new over old probability of an action

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Model?</div></th><th nowrap><div align="left">Learns</div></th><th nowrap><div align="left">Data from</div></th><th nowrap><div align="left">Depth</div></th><th nowrap><div align="left">Bias of the weight or target</div></th><th nowrap><div align="left">Variance</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">dynamic programming</div></td><td nowrap><div align="left">yes</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">none (planning)</div></td><td nowrap><div align="left">1 step, all successors</div></td><td nowrap><div align="left">uses the current estimate</div></td><td nowrap><div align="left">none from sampling</div></td></tr>
<tr><td nowrap><div align="left">Monte Carlo</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">policy evaluated</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">high</div></td></tr>
<tr><td nowrap><div align="left">TD / Q-learning / SARSA / DQN</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">values</div></td><td nowrap><div align="left">on-policy (SARSA) or off-policy</div></td><td nowrap><div align="left">1 step ($p$-step between)</div></td><td nowrap><div align="left">while the estimate is wrong (Q-learning and DQN add <i>maximization bias</i>, from the max)</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy</div></td><td nowrap><div align="left">on-policy</div></td><td nowrap><div align="left">to the end</div></td><td nowrap><div align="left">none</div></td><td nowrap><div align="left">high (lower with a baseline, still high)</div></td></tr>
<tr><td nowrap><div align="left">actor-critic</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy and value</div></td><td nowrap><div align="left">on-policy</div></td><td nowrap><div align="left">1 step ($p$-step or GAE between)</div></td><td nowrap><div align="left">from the critic</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">PPO</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy and value</div></td><td nowrap><div align="left">near on-policy (ratio, clip)</div></td><td nowrap><div align="left">GAE</div></td><td nowrap><div align="left">from the critic (clipping limits the step, not the weight)</div></td><td nowrap><div align="left">lower</div></td></tr>
<tr><td nowrap><div align="left">GRPO</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy</div></td><td nowrap><div align="left">near on-policy (group per prompt)</div></td><td nowrap><div align="left">whole response</div></td><td nowrap><div align="left">a shrink by (G-1)/G ($G$ completions per prompt) and a rescaling by the standard deviation</div></td><td nowrap><div align="left">noisier baseline, no critic</div></td></tr>
<tr><td nowrap><div align="left">RLHF</div></td><td nowrap><div align="left">no</div></td><td nowrap><div align="left">policy (reward from a reward model)</div></td><td nowrap><div align="left">on-policy samples</div></td><td nowrap><div align="left">as PPO</div></td><td nowrap><div align="left">the reward model is a proxy</div></td><td nowrap><div align="left">as PPO</div></td></tr>
</tbody></table>


## The weight ladder (Part B)

**Why this section?** The methods of Part B, and GRPO of Part C, are one theorem. They differ in the weight, the step and the baseline source.

**Part B in two tables.** This slide: the ladder, the changes to the weight in order. Next slide: the map, the methods those changes produce, with the step and the baseline source.

**Four changes to the weight, in order. The first three leave the gradient unbiased. Only the fourth trades bias for lower variance.**

<table>
<thead><tr><th><div align="left">change</div></th><th><div align="left">weight multiplying the score function</div></th><th><div align="left">why</div></th><th><div align="left">bias</div></th><th><div align="left">variance</div></th><th><div align="left">needs a critic</div></th></tr></thead>
<tbody>
<tr><td><div align="left"><b>1. theorem</b> (B1)</div></td><td><div align="left">$\rlret_{0,\traj}$</div></td><td><div align="left">the theorem's weight: the return of the whole episode, the same for every step</div></td><td><div align="left">none</div></td><td><div align="left">high: $|\traj|$ random rewards</div></td><td><div align="left">no</div></td></tr>
<tr><td><div align="left"><b>2. reward-to-go</b> (B2)</div></td><td><div align="left">$\color{#D95F02}{\rlret_{\tt,\traj}}$</div></td><td><div align="left">rewards before step $\tt$ cannot depend on action $\tt$: they add only noise</div></td><td><div align="left">none</div></td><td><div align="left">lower: $|\traj|-\tt$ random rewards</div></td><td><div align="left">no</div></td></tr>
<tr><td><div align="left"><b>3. baseline</b> (B2, B3)</div></td><td><div align="left">$\rlret_{\tt,\traj} \color{#D95F02}{- b(\stateseq_{\tt,\traj})}$</div></td><td><div align="left">subtract what every action in the state shares; the best choice is the state value, which a critic estimates (B3)</div></td><td><div align="left">none</div></td><td><div align="left">a little lower: the same rewards, minus a shared offset</div></td><td><div align="left">no with a moving average; yes with a critic</div></td></tr>
<tr><td><div align="left"><b>4. bootstrap</b> (B3)</div></td><td><div align="left">$\color{#D95F02}{\rewseq_{\tt+1,\traj} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1,\traj})} - \statevalfun_\phi(\stateseq_{\tt,\traj})$</div></td><td><div align="left">replace the sampled tail of the return by the critic's estimate</div></td><td><div align="left"><b><span style="color:#b3261e">yes</span></b></div></td><td><div align="left"><b>much lower: one random reward and one next state</b></div></td><td><div align="left">yes, as a bootstrap</div></td></tr>
</tbody>
</table>

- orange: the part that changed from the row above
- $\statevalfun_\phi$ is the critic, defined in B3
- the dial between rows 3 and 4 is $p$-step TD or GAE (B3)

**The method map: the same score function, three choices**

**Same score function in every method. The methods differ in three choices: the weight, how far one update moves, and where the baseline comes from.**

> **A baseline adds no bias. A bootstrap adds bias and lowers the variance further. The group mean trades the critic's bias for the noise of a finite group.**

<table>
<thead><tr><th></th>
<th><div align="left">REINFORCE, over-simplified (B1)</div></th>
<th><div align="left">REINFORCE with baseline (B2)</div></th>
<th><div align="left">actor-critic (B3)</div></th>
<th><div align="left">PPO (B4)</div></th>
<th><div align="left">GRPO (C2)</div></th>
</tr></thead>
<tbody>
<tr><td><div align="left"><b>weight (slot 1)</b></div></td><td><div align="left">$\rlret_{0,\traj}$</div></td><td><div align="left">$\color{#D95F02}{\rlret_{\tt,\traj} - b}$</div></td><td><div align="left">$\color{#D95F02}{\text{TD error or GAE}}$</div></td><td><div align="left">GAE, learned critic</div></td><td><div align="left">group-relative advantage</div></td></tr>
<tr><td><div align="left"><b>how far (slot 2)</b></div></td><td><div align="left">plain step</div></td><td><div align="left">plain step</div></td><td><div align="left">plain step</div></td><td><div align="left"><b>clipped ratio, $K$ epochs per batch</b></div></td><td><div align="left">clipped ratio, as PPO</div></td></tr>
<tr><td><div align="left"><b>baseline from (slot 3)</b></div></td><td><div align="left">none</div></td><td><div align="left"><b>moving average</b> of past returns</div></td><td><div align="left"><b>learned critic</b></div></td><td><div align="left">learned critic</div></td><td><div align="left"><b>mean of a group</b> of episodes</div></td></tr>
<tr><td><div align="left"><b>bias</b></div></td><td><div align="left">none</div></td><td><div align="left">none</div></td><td><div align="left"><b><span style="color:#b3261e">the critic's</span></b></div></td><td><div align="left">GAE's, plus the surrogate and the clip</div></td><td><div align="left">none from a critic; noise from a finite group</div></td></tr>
<tr><td><div align="left"><b>variance</b></div></td><td><div align="left">highest</div></td><td><div align="left">lower</div></td><td><div align="left">lowest</div></td><td><div align="left">GAE's, ratio kept near 1</div></td><td><div align="left">falls as the group grows</div></td></tr>
<tr><td><div align="left"><b>extra cost</b></div></td><td><div align="left">none</div></td><td><div align="left">none</div></td><td><div align="left">a second network to train</div></td><td><div align="left">$K$ passes over each batch</div></td><td><div align="left">a group of episodes per prompt; no critic</div></td></tr>
</tbody>
</table>

- bold: the choice that is new in that method
- the first method is B1's estimator, with the whole-episode return
- the second adds reward-to-go and a baseline (B2)
- Sutton and Barto's REINFORCE (reward-to-go, no baseline) is the weight of ladder row 2 alone (the ladder is the other table)

**Reading the map: bias, variance and the extra costs**

- a TD error uses the critic's number in place of the sampled tail of the return (B3)
- a critic used as the baseline only adds no bias, however wrong it is
    - it lowers the variance if it is close
- with $p$-step returns or GAE the critic's bias falls as $p$ or $\lambda$ grows
- $K$: passes over each batch (B4)
- the surrogate: the objective PPO maximizes in place of $J$ (B4)
- the ladder is in B1 to B3; the full map is in B4 and C2

- in the grid, all the map's rows are in the middle and right columns, near on-policy
- RLHF-style recipes that use PPO or GRPO make choices from the map
    - they add a reward source (program, reward model, judge)
    - and usually a KL penalty to a reference model (its equation is on the second equations slide)

## The threads, one last time

**Why this section?** Each thread appeared early and came back. This is the whole path of each.

> **A thread is a theme that returns in several sessions, each time in a new form. Read the arrow as "comes back as", in session order.**

<table>
<thead><tr><th nowrap><div align="left">Thread</div></th><th nowrap><div align="left">The question it keeps asking</div></th><th nowrap><div align="left">Its path, from the first session to the last</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>bias and variance</b></div></td><td nowrap><div align="left">how noisy, and how biased, is the number we learn from?</div></td><td nowrap><div align="left">the targets of Monte Carlo and TD $\to$ the weight in the policy gradient (the ladder: reward-to-go, baseline, bootstrap) $\to$ baseline, critic, group mean (clipping, separately, limits the step)</div></td></tr>
<tr><td nowrap><div align="left"><b>backup width and depth</b></div></td><td nowrap><div align="left">how many steps ahead, and how many branches, does one update use?</div></td><td nowrap><div align="left">dynamic programming, Monte Carlo, $p$-step TD $\to$ return, TD error, GAE</div></td></tr>
<tr><td nowrap><div align="left"><b>on-policy versus off-policy</b></div></td><td nowrap><div align="left">whose samples does the update learn from?</div></td><td nowrap><div align="left">light at Monte Carlo, defined at SARSA versus Q-learning $\to$ replay allowed in DQN $\to$ the ratio in PPO</div></td></tr>
<tr><td nowrap><div align="left"><b>evaluate versus improve</b></div></td><td nowrap><div align="left">how do we alternate judging a policy and changing it?</div></td><td nowrap><div align="left">policy iteration $\to$ control with values $\to$ the critic and the actor</div></td></tr>
<tr><td nowrap><div align="left"><b>exploration</b></div></td><td nowrap><div align="left">how does the agent try things it has not tried?</div></td><td nowrap><div align="left">greedy versus $\epsilon$-greedy $\to$ a stochastic policy that explores by sampling $\to$ a group that must differ</div></td></tr>
<tr><td nowrap><div align="left"><b>the proxy problem</b> (new in Part C)</div></td><td nowrap><div align="left">does the reward measure what we want?</div></td><td nowrap><div align="left">only in Part C: the reward is a stand-in, so it can be exploited (reward hacking); the KL penalty to a reference model limits how far</div></td></tr>
</tbody></table>


# Equations behind the ideas (1): value-based

**Why this section?** Each idea of Part A condensed to its equation.

> **Do not confuse two things. $\rlret_\tt$ is the return of one specific trajectory: a number. $\statevalfun_\pi(\stateseq_\tt)$ is the average of that return over all trajectories from the state.**

- a new trajectory gives a new $\rlret_\tt$; $\statevalfun_\pi$ is fixed for the policy
- the return recursion is exact: it holds on every trajectory
- the Bellman equation averages that recursion over trajectories
- TD looks the same but replaces the actual tail $\rlret_{\tt+1}$ with the *estimate* $\hat{\statevalfun}(\stateseq_{\tt+1})$: that is the bootstrap
- Part B writes $\rlret_{\tt,\traj}$ to name the trajectory
- a hat marks an estimate: TD updates $\hat{\statevalfun}$ toward targets; $\statevalfun_\pi$, no hat, is the true value
    - a subscript can also mark an estimate: the critic $\statevalfun_\phi$ of Part B
    - Sutton and Barto write $V$ and $Q$ for the estimates

<table>
<thead><tr><th nowrap><div align="left">Idea</div></th><th nowrap><div align="left">The equation</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">the return</div></td><td nowrap><div align="left">$\rlret_\tt = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1}$</div></td></tr>
<tr><td nowrap><div align="left">the Bellman equation</div></td><td nowrap><div align="left">$\statevalfun_\pi(\state) = \sum_\act \pi(\act \mid \state) \sum_{\state', \rew} \transp(\state', \rew \mid \state, \act) \bigl( \rew + \disc \, \statevalfun_\pi(\state') \bigr)$</div></td></tr>
<tr><td nowrap><div align="left">TD learning</div></td><td nowrap><div align="left">$\hat{\statevalfun}(\stateseq_\tt) \leftarrow \hat{\statevalfun}(\stateseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \hat{\statevalfun}(\stateseq_{\tt+1}) - \hat{\statevalfun}(\stateseq_\tt) \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">SARSA</div></td><td nowrap><div align="left">$\hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) \leftarrow \hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \hat{\actvalfun}(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">Q-learning</div></td><td nowrap><div align="left">$\hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) \leftarrow \hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \max{\act'} \hat{\actvalfun}(\stateseq_{\tt+1}, \act') - \hat{\actvalfun}(\stateseq_\tt, \actseq_\tt) \bigr]$</div></td></tr>
</tbody></table>


# Equations behind the ideas (2): policy-based and language models

**Why this section?** Part B and Part C in the same form. The advantage and the ratio are defined in their sessions. Here they are named.

<table>
<thead><tr><th nowrap><div align="left">Idea</div></th><th nowrap><div align="left">The equation</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">the policy gradient (reward-to-go form: ladder row 2)</div></td><td nowrap><div align="left">$\nabla_\theta J(\theta) = \Exp{\traj \sim \pi_\theta} \bigl[ \sum_\tt \nabla_\theta \log \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, \rlret_{\tt,\traj} \bigr]$</div></td></tr>
<tr><td nowrap><div align="left">the advantage form</div></td><td nowrap><div align="left">the same with the weight $\hat{\advseq}_\tt$ in place of $\rlret_{\tt,\traj}$: the return minus a baseline, or an estimate such as the TD error $\delta_\tt = \rewseq_{\tt+1} + \disc \, \statevalfun_\phi(\stateseq_{\tt+1}) - \statevalfun_\phi(\stateseq_\tt)$ or GAE</div></td></tr>
<tr><td nowrap><div align="left">the probability ratio</div></td><td nowrap><div align="left">$r_\tt(\theta) = \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, / \, \pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)$</div></td></tr>
<tr><td nowrap><div align="left">PPO (clipped objective)</div></td><td nowrap><div align="left">$\min{} \bigl( r_\tt \, \hat{\advseq}_\tt, \; \text{clip}(r_\tt, \, 1 - \epsilon_{\text{clip}}, \, 1 + \epsilon_{\text{clip}}) \, \hat{\advseq}_\tt \bigr)$, with $\hat{\advseq}_\tt$ an <i>estimate</i> of the advantage ($\epsilon_{\text{clip}}$ is not the $\epsilon$ of $\epsilon$-greedy)</div></td></tr>
<tr><td nowrap><div align="left">GRPO advantage</div></td><td nowrap><div align="left">$\hat{\advseq}_i = (\rew_i - \text{mean}) / \text{std}$ over a group of completions, in the same clipped objective ($i$ indexes the group, as in the GRPO paper)</div></td></tr>
<tr><td nowrap><div align="left">KL penalty (language models)</div></td><td nowrap><div align="left">subtract $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, $\beta$ the weight, $\pi_{\text{ref}}$ a frozen reference model: a per-token reward in PPO, a loss term in GRPO</div></td></tr>
</tbody></table>


## Choosing a family

**Why this section?** From knowing the methods to choosing among them for a new problem.

> **The grid's three questions decide: do we have a model, what must we learn (values or a policy), and can the data be reused (on- or off-policy)?**

- **the model is known and small** (a board game's rules, a grid): dynamic programming
- **small discrete states and actions, model unknown:** tabular Q-learning or SARSA
- **large or continuous states, a few discrete actions:** DQN
- **continuous actions, or a stochastic policy is needed:** policy-gradient methods
    - PPO is a common on-policy default
    - off-policy methods such as SAC (soft actor-critic) also serve continuous control; not covered here
- **language models:** the policy is the language model
    - the reward comes from a program, a reward model or a judge
    - PPO and GRPO are the tools
- **only fixed data, no environment:** offline methods (left out of this series)
- **samples are costly or unsafe:** learn or use a model, or offline data (not covered)
- in every case
    - reward design decides what the agent learns (reward hacking)
    - exploration decides what *can* be learned (one view, not established here: RL mostly raises the probability of what the policy already tries)


## What this series left out

**Why this section?** A map should show its edges.

- **learning a model** of the environment and planning with it (model-based RL with a learned model)
- **off-policy actor-critic methods** for continuous actions (the empty cell in the grid)
- **continuous-action value methods and distributional or ensemble variants** of DQN
- **the theory** behind convergence of these methods, beyond the tabular case
- **offline RL**
- **natural-gradient methods**: updates that follow the geometry of the policy distribution rather than raw parameters
    - and trust-region theory (TRPO, trust-region policy optimization) beyond the short mention
- **maximum-entropy methods**, **importance-sampling corrections** for multi-step off-policy learning
- **imitation learning** and **hierarchical RL**
- **multi-agent and partially observed problems**, beyond what the examples showed
- **the language-model details** (reward-model training at scale, RLHF at scale, GRPO variants, the derivation and use of DPO): the Advanced course


# What we established, and what it sets up

- **Established**
    - one loop, from Frozen Lake and Microduck to language models
    - three questions place every method: model, what is learned, which policy generates the data
    - the same few threads run through both the value-based and the policy-gradient methods
        - bias and variance, backup depth, on- and off-policy, evaluate versus improve, exploration
    - Part C added the proxy problem
    - Part B's methods differ in three choices (the weight, how far, the baseline source), shown by the ladder and the map
    - a method family can be chosen from the three questions
- **Sets up**
    - the Advanced course: reward models, RLHF, GRPO with verifiable rewards, preference optimization
    - the code notebooks and lab exercises that implement these methods
